# Open-XTTS-Trainer: End-to-End 3-Stage Training & Proof Verification Demo

This notebook runs an end-to-end training verification on **5 minutes of clean human speech** across all 3 stages of **XTTS v2**:
1. **Stage 2:** Text Tokenizer & Vocabulary Extension for New Scripts + Autoregressive GPT Backbone Training (Real DVAE Code Cross-Entropy)
2. **Stage 3:** Latent HiFi-GAN Vocoder Fine-Tuning (True Temporal Latents + Pretrained Weights)
3. **Stage 1:** Discrete VAE (DVAE) Audio Tokenizer Fine-Tuning at native 22,050 Hz
4. **Synthesis & Proof Export:** Audio generation from trained checkpoints + 1-click export of verified logs (`open_xtts_proof_logs.zip`).

## 1. Setup Environment & Install Dependencies

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'

# Check GPU availability
!nvidia-smi

# Install dependencies and Idiap's Coqui TTS fork
!pip install -q git+https://github.com/idiap/coqui-ai-TTS.git
!pip install -q torchcodec
!git clone https://github.com/amitbhakt/open-xtts-trainer.git 2>/dev/null || (cd open-xtts-trainer && git pull)
%cd open-xtts-trainer
!pip install -q -e .
!mkdir -p logs checkpoints xtts_base

## 2. Download Pretrained XTTS-v2 Base Checkpoints & 5-Min Speech Dataset

In [ ]:
# 1. Download official base checkpoints (vocab, config, dvae, model)
!wget -q -O xtts_base/vocab.json https://huggingface.co/coqui/XTTS-v2/raw/main/vocab.json
!wget -q -O xtts_base/config.json https://huggingface.co/coqui/XTTS-v2/raw/main/config.json
!wget -q -O xtts_base/dvae.pth https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth
!wget -q -O xtts_base/model.pth https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth

# 2. Download 5.0 Minutes of Clean Human Speech (100 segmented clips)
!python -m xtts_trainer.speech_data --minutes 5 --output_dir sample_audio
!ls -lh sample_audio | head -n 10

## 3. Stage 2: GPT Alignment & Vocabulary Extension

In [ ]:
# 1. Sample target text containing new language script
with open("sample_target.txt", "w", encoding="utf-8") as f:
    f.write("ދިވެހި ބަސް. สวัสดี. مرحبا. नमस्ते")

# 2. Extend tokenizer vocab (automatically resizes text_embedding & text_head)
!python xtts_trainer/stage2_gpt/vocab_extender.py --base_vocab xtts_base/vocab.json --target_text sample_target.txt --output_vocab xtts_base/vocab_extended.json

# 3. Fine-Tune Autoregressive GPT Backbone with true DVAE acoustic target cross-entropy
# Output is logged to logs/stage2_gpt.log for automated verification
!python xtts_trainer/stage2_gpt/train.py \
    --audio_dir sample_audio \
    --vocab_path xtts_base/vocab_extended.json \
    --checkpoint_dir xtts_base \
    --output_dir checkpoints/gpt \
    --epochs 5 \
    --batch_size 4 \
    --device cuda 2>&1 | tee logs/stage2_gpt.log

## 4. Stage 3: Latent HiFi-GAN Vocoder Fine-Tuning

In [ ]:
# 1. Extract true per-frame temporal GPT latents & speaker embeddings (~1 min on T4 GPU)
!python xtts_trainer/stage3_hifigan/extract_latents.py \
    --audio_dir sample_audio \
    --output_dir extracted_latents \
    --checkpoint_dir xtts_base \
    --device cuda 2>&1 | tee logs/extract_latents.log

# 2. Fine-tune Latent HiFi-GAN Vocoder starting from pretrained base weights (25 Epochs)
!python xtts_trainer/stage3_hifigan/train.py \
    --audio_dir sample_audio \
    --latents_dir extracted_latents \
    --checkpoint_dir xtts_base \
    --output_dir checkpoints/hifigan \
    --batch_size 16 \
    --epochs 25 \
    --device cuda 2>&1 | tee logs/stage3_hifigan.log

## 5. Stage 1: Discrete VAE (DVAE) Fine-Tuning

In [ ]:
# Run native DVAE fine-tuning at 22,050 Hz (25 Epochs)
!python xtts_trainer/stage1_dvae/train.py \
    --audio_dir sample_audio \
    --output_dir checkpoints/dvae \
    --continue_path xtts_base/dvae.pth \
    --batch_size 16 \
    --epochs 25 \
    --device cuda 2>&1 | tee logs/stage1_dvae.log

## 6. Test Speech Synthesis with Custom Trained Weights

In [ ]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
import IPython.display as ipd
import soundfile as sf

# Load model using fine-tuned Stage 2 GPT checkpoint
config = XttsConfig()
if os.path.exists("xtts_base/config.json"):
    config.load_json("xtts_base/config.json")

model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_path="checkpoints/gpt/model.pth",
    vocab_path="checkpoints/gpt/vocab.json",
    eval=True
)
model.cuda()

print("Synthesizing voice clone with fine-tuned checkpoint...")
outputs = model.synthesize(
    "Artificial intelligence is rapidly transforming how we communicate and build open technology every day.",
    config,
    speaker_wav="sample_audio/speech_001.wav",
    gpt_cond_len=6,
    language="en",
)
sf.write("cloned_speech.wav", outputs["wav"], 24000)
print("Audio synthesized successfully! Playing output:")
display(ipd.Audio("cloned_speech.wav"))


## 7. Generate Verification Proof Report & 1-Click Download

In [ ]:
# 1. Run automated proof compiler across all execution logs
!python scripts/generate_proof_report.py --logs_dir logs --checkpoints_dir checkpoints --audio_file cloned_speech.wav

# 2. Render Markdown proof table directly in Colab
from IPython.display import Markdown, display
with open("logs/training_proof_report.md", "r") as f:
    display(Markdown(f.read()))

# 3. Trigger 1-click browser download of the complete proof archive
try:
    from google.colab import files
    print("Downloading proof archive (open_xtts_proof_logs.zip)...")
    files.download('open_xtts_proof_logs.zip')
except Exception:
    print("Archive available at: open_xtts_proof_logs.zip")
